# CircuitLM Training ??? Personal Dataset

Trains a hybrid CircuitLM (PDA circuit + neural corrector) on your personal conversations.
Runs on Kaggle GPU. Uses ALL personal data merged and deduplicated (~24K lines, 7.4 MB).

**BEFORE RUNNING:**
1. Go to **kaggle.com ??? Datasets ??? Create New Dataset**
2. Upload **`all_personal_training.txt`** from the circuit_lm repo on GitHub
   (GitHub path: `circuit_lm/all_personal_training.txt`)
3. Name it `circuit-lm-personal`
4. Open this notebook ??? **Add Input** ??? select `circuit-lm-personal`

**Runtime:** GPU (P100/T4). ~15-30 min.

## 1. Install Dependencies

In [ ]:
%pip install orjson sentencepiece torch --quiet
print('Installed')

## 2. Load Data (from Kaggle dataset input)

In [ ]:
import os

# Find all_personal_training.txt ??? the merged + deduplicated file
possible_paths = [
    '/kaggle/input/circuit-lm-personal/all_personal_training.txt',
    '/kaggle/input/circuitlm-personal/all_personal_training.txt',
    './all_personal_training.txt',
    # Fallback to individual files
    '/kaggle/input/circuit-lm-personal/research_evolver_data.txt',
    '/kaggle/input/circuit-lm-personal/chatgpt_data.txt',
    '/kaggle/input/circuit-lm-personal/marble_data.txt',
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError(
        f'Data file not found.\n'
        f'Upload all_personal_training.txt to Kaggle: '
        f'github.com/toxzak-svg/circuit_lm/blob/master/all_personal_training.txt'
    )

with open(data_path, encoding='utf-8', errors='replace') as f:
    text = f.read()

lines = text.strip().split('\n')
print(f'Loaded {len(lines)} lines ({len(text)/1024:.0f} KB) from {os.path.basename(data_path)}')

# Use all available data for training
train_text = text
print(f'Using ALL {len(lines)} lines for training')

## 3. Clone CircuitLM and Add to Path

In [ ]:
!git clone https://github.com/toxzak-svg/circuit_lm.git /tmp/circuit_lm
import sys
sys.path.insert(0, '/tmp/circuit_lm/src')
print('circuit_lm cloned and in path')

## 4. Tokenizer + Data Prep

In [ ]:
import time
from circuit_lm.tokenizer import Tokenizer
from circuit_lm.data import load_sequences

VOCAB_SIZE = 4096

print('Building BPE tokenizer...')
t0 = time.time()
tokenizer = Tokenizer.from_text(
    train_text,
    vocab_size=VOCAB_SIZE,
    mode='bpe',
    bpe_merges=VOCAB_SIZE,
)
print(f'  vocab={tokenizer.vocab_size}, took {time.time()-t0:.1f}s')

print('Tokenizing sequences...')
t0 = time.time()
sequences = load_sequences(train_text, tokenizer)
print(f'  {len(sequences)} sequences, {sum(len(s) for s in sequences)} tokens, took {time.time()-t0:.1f}s')

## 5. Train PDA Circuit (CP-SAT)

In [ ]:
from circuit_lm.train_joint_pda_cpsat import train_joint_pda as train_pda
import time

STATE_BITS = 6     # 64 states
STACK_DEPTH = 4
STACK_STEPS = 15
TRANS_STEPS = 22
EMIT_STEPS = 23   # total = 60

print('Training PDA circuit...')
t0 = time.time()
circuit = train_pda(
    sequences=sequences,
    vocab_size=tokenizer.vocab_size,
    state_bits=STATE_BITS,
    stack_depth=STACK_DEPTH,
    stack_steps=STACK_STEPS,
    transition_steps=TRANS_STEPS,
    emission_steps=EMIT_STEPS,
)
circuit_time = time.time() - t0
print(f'  Circuit trained in {circuit_time:.1f}s')

## 6. Save Circuit + Tokenizer

In [ ]:
from circuit_lm.io import save_model

CIRCUIT_PATH = '/tmp/circuit_4k.json'
save_model(circuit, tokenizer, CIRCUIT_PATH)
import os
print(f'Circuit saved: {os.path.getsize(CIRCUIT_PATH)/1024:.0f} KB')

## 7. Train Hybrid Corrector

In [ ]:
import time
import torch
from hybrid import train_hybrid

print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

CORRECTOR_PATH = '/tmp/corrector_4k.pt'
DATA_PATH = '/tmp/train_text.txt'

with open(DATA_PATH, 'w') as f:
    f.write(train_text)

print('Training hybrid corrector...')
t0 = time.time()
corrector = train_hybrid(
    circuit_path=CIRCUIT_PATH,
    data_path=DATA_PATH,
    output_path=CORRECTOR_PATH,
    num_epochs=5,
    batch_size=128,
    circuit_weight=0.5,
    max_examples=80000,
)
corrector_time = time.time() - t0
print(f'Corrector trained in {corrector_time:.1f}s')

## 8. Download Results

In [ ]:
from IPython.display import FileLink
print('Circuit (circuit_4k.json):')
FileLink(CIRCUIT_PATH)
print('Corrector (corrector_4k.pt):')
FileLink(CORRECTOR_PATH)
print()
print(f'Total training time: {circuit_time + corrector_time:.1f}s')